# Assignment 1
# Andrew Silveira
# 5077086

### Imports

In [7]:
import numpy as np
# --- PATCH START ---
# This fixes the "AttributeError: module 'numpy' has no attribute 'bool8'"
# by manually re-adding the missing attribute to NumPy.
if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_
# --- PATCH END ---

import gym
import random
from collections import deque
import sys
import torch
import torch.nn as nn
import torch.optim as optim

from assignment3_utils import process_frame, transform_reward

In [8]:
class DQNCNN(nn.Module):
    def __init__(self, action_size):
        super(DQNCNN, self).__init__()
        
        # Input: (batch, 4, 84, 80)
        
        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU()
        )
        
        # Compute conv output size
        
        dummy = torch.zeros(1, 4, 84, 80)
        conv_out = self.conv(dummy).view(1, -1).size(1)
        
        self.fc = nn.Sequential(
            nn.Linear(conv_out, 512),
            nn.ReLU(),
            nn.Linear(512, action_size)
        )
    
    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

In [9]:
class ReplayBuffer:
    def __init__(self, capacity=50000):
        self.capacity = capacity
        #self.buffer = []
        #self.position = 0
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):     
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            np.array(states),
            np.array(actions),
            np.array(rewards),
            np.array(next_states),
            np.array(dones)
        )

    def __len__(self):
        return len(self.buffer)

In [10]:
class DQNAgent:
    def __init__(self, action_size, batch_size=8, gamma=0.95, lr=1e-4):
        self.action_size = action_size
        self.batch_size = batch_size
        self.gamma = gamma

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.policy_net = DQNCNN(action_size).to(self.device)
        self.target_net = DQNCNN(action_size).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.memory = ReplayBuffer()

        # Exploration
        self.epsilon = 1.0
        self.epsilon_min = 0.05
        self.epsilon_decay = 0.995

    def act(self, state):
        if random.random() < self.epsilon:
            return random.randrange(self.action_size)

        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        q_values = self.policy_net(state)
        return torch.argmax(q_values).item()

    def train_step(self):
        if len(self.memory) < self.batch_size:
            return

        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)

        states = torch.FloatTensor(states).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        actions = torch.LongTensor(actions).to(self.device)
        rewards = torch.FloatTensor(rewards).to(self.device)
        dones = torch.FloatTensor(dones).to(self.device)

        q_values = self.policy_net(states)
        next_q_values = self.target_net(next_states)

        q_target = rewards + self.gamma * torch.max(next_q_values, dim=1)[0] * (1 - dones)

        q_pred = q_values.gather(1, actions.unsqueeze(1)).squeeze()

        loss = nn.MSELoss()(q_pred, q_target.detach())

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # Decay epsilon
        self.epsilon = max(self.epsilon * self.epsilon_decay, self.epsilon_min)

In [11]:
def stack_frames(stacked_frames, new_frame, is_new_episode):
    if is_new_episode:
        stacked_frames = deque([new_frame] * 4, maxlen=4)
    else:
        stacked_frames.append(new_frame)

    return np.stack(stacked_frames, axis=0), stacked_frames

def train_pong(batch_size=8, target_update_rate=10, num_episodes=500):
    #env = gym.make("PongDeterministic-v4")
    env = gym.make("PongDeterministic-v4", render_mode="human")
    action_size = env.action_space.n

    agent = DQNAgent(action_size, batch_size=batch_size)

    scores = []
    last5_rewards = []

    for episode in range(num_episodes):
        obs, info = env.reset()
        frame = process_frame(obs, (84, 80))
        stacked_frames = None
        state, stacked_frames = stack_frames(stacked_frames, frame, True)

        done = False
        total_reward = 0

        while not done:
            action = agent.act(state)
            next_obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            reward = transform_reward(reward)
            next_frame = process_frame(next_obs, (84, 80))
            next_state, stacked_frames = stack_frames(stacked_frames, next_frame, False)

            agent.memory.push(state, action, reward, next_state, done)
            agent.train_step()

            state = next_state
            total_reward += reward

        scores.append(total_reward)
        last5_rewards.append(np.mean(scores[-5:]))

        print(f"Episode {episode} | Score: {total_reward} | Avg(5): {last5_rewards[-1]:.2f}", flush=True)

        # Update target network every N episodes
        if episode % target_update_rate == 0:
            agent.target_net.load_state_dict(agent.policy_net.state_dict())

    env.close()
    return scores, last5_rewards

In [12]:
if __name__ == "__main__":
    scores, avg5 = train_pong(
        batch_size=8,
        target_update_rate=10,
        num_episodes=15
    )

Episode 0 | Score: -20.0 | Avg(5): -20.00
Episode 1 | Score: -21.0 | Avg(5): -20.50
Episode 2 | Score: -21.0 | Avg(5): -20.67
Episode 3 | Score: -21.0 | Avg(5): -20.75
Episode 4 | Score: -18.0 | Avg(5): -20.20
Episode 5 | Score: -20.0 | Avg(5): -20.20
Episode 6 | Score: -20.0 | Avg(5): -20.00
Episode 7 | Score: -20.0 | Avg(5): -19.80
Episode 8 | Score: -21.0 | Avg(5): -19.80
Episode 9 | Score: -20.0 | Avg(5): -20.20
Episode 10 | Score: -19.0 | Avg(5): -20.00
Episode 11 | Score: -21.0 | Avg(5): -20.20
Episode 12 | Score: -21.0 | Avg(5): -20.40
Episode 13 | Score: -20.0 | Avg(5): -20.20
Episode 14 | Score: -20.0 | Avg(5): -20.20
